<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-12-production-deploy/lesson-12.1-infra-setup/notebooks/GCP_Capstone_12.1_InfraSetup.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.1 Infrastructure & Project Setup
**Netsetos GenAI Engineering — GCP Capstone**

Terraform the foundation: VPC + Artifact Registry + Secret Manager + Firestore + GCS + Cloud Run service accounts + IAP brand + budget alerts. Zero console clicks from here.

## Cell 1: Enable the APIs (one gcloud command)

In [ ]:
PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS
REGION = 'us-central1'
INDIA_REGION = 'asia-south1'

ENABLE_APIS = f'''
gcloud config set project {PROJECT_ID}

gcloud services enable \\
  run.googleapis.com \\
  compute.googleapis.com \\
  vpcaccess.googleapis.com \\
  pubsub.googleapis.com \\
  artifactregistry.googleapis.com \\
  secretmanager.googleapis.com \\
  firestore.googleapis.com \\
  storage.googleapis.com \\
  aiplatform.googleapis.com \\
  documentai.googleapis.com \\
  speech.googleapis.com \\
  texttospeech.googleapis.com \\
  dlp.googleapis.com \\
  iap.googleapis.com \\
  iamcredentials.googleapis.com \\
  cloudbuild.googleapis.com \\
  cloudtrace.googleapis.com \\
  monitoring.googleapis.com \\
  logging.googleapis.com \\
  billingbudgets.googleapis.com
'''
print(ENABLE_APIS)

## Cell 2: Terraform backend.tf + provider.tf

In [ ]:
BACKEND_TF = '''
terraform {
  required_version = ">= 1.9.0"
  required_providers {
    google      = { source = "hashicorp/google",      version = "~> 6.15" }
    google-beta = { source = "hashicorp/google-beta", version = "~> 6.15" }
  }
  backend "gcs" {
    prefix = "documind/env"
  }
}

provider "google" {
  project = var.project_id
  region  = var.region
}
provider "google-beta" {
  project = var.project_id
  region  = var.region
}
'''
with open('backend.tf', 'w') as f: f.write(BACKEND_TF)
print('backend.tf written')
print()
print('One-time: create the state bucket (NOT managed by tf itself)')
print(f'  gsutil mb -l {REGION} -b on gs://{PROJECT_ID}-tfstate')
print(f'  gsutil versioning set on gs://{PROJECT_ID}-tfstate')

## Cell 3: variables.tf + service account module

In [ ]:
VARS_TF = '''
variable "project_id"  { type = string }
variable "region"      {
  type    = string
  default = "us-central1"
}
variable "india_region"{
  type    = string
  default = "asia-south1"
}
variable "env"         {
  type    = string
  default = "dev"
}  # dev | staging | prod
variable "admin_emails"{ type = list(string) }
'''

SA_TF = '''
# Three service accounts, one per service, least-privilege from day zero.
resource "google_service_account" "ui" {
  account_id   = "documind-ui-sa"
  display_name = "DocuMind UI (Streamlit on Cloud Run)"
}
resource "google_service_account" "api" {
  account_id   = "documind-api-sa"
  display_name = "DocuMind API (FastAPI backend)"
}
resource "google_service_account" "admin" {
  account_id   = "documind-admin-sa"
  display_name = "DocuMind Admin + Observability"
}

# Signed URLs on Cloud Run need self-impersonation
resource "google_service_account_iam_member" "ui_self_impersonate" {
  service_account_id = google_service_account.ui.name
  role               = "roles/iam.serviceAccountTokenCreator"
  member             = "serviceAccount:${google_service_account.ui.email}"
}

locals {
  ui_roles = [
    "roles/aiplatform.user",
    "roles/documentai.apiUser",
    "roles/secretmanager.secretAccessor",
    "roles/datastore.user",
    "roles/speech.editor",
  ]
  api_roles = [
    "roles/aiplatform.user",
    "roles/datastore.user",
    "roles/secretmanager.secretAccessor",
    "roles/logging.logWriter",
    "roles/cloudtrace.agent",
  ]
  admin_roles = [
    "roles/monitoring.viewer",
    "roles/logging.viewer",
    "roles/datastore.user",
    "roles/bigquery.jobUser",
    "roles/run.developer",   # budget-guard fn calls run_v2.update_service to set min_instances=0
  ]
}

resource "google_project_iam_member" "ui" {
  for_each = toset(local.ui_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_project_iam_member" "api" {
  for_each = toset(local.api_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.api.email}"
}
resource "google_project_iam_member" "admin" {
  for_each = toset(local.admin_roles)
  project  = var.project_id
  role     = each.value
  member   = "serviceAccount:${google_service_account.admin.email}"
}
'''
with open('variables.tf', 'w') as f: f.write(VARS_TF)
with open('sa.tf', 'w') as f: f.write(SA_TF)
print('variables.tf + sa.tf written')

## Cell 4: VPC + Artifact Registry + Firestore

In [ ]:
NETWORK_TF = '''
resource "google_compute_network" "vpc" {
  name                    = "documind-vpc"
  auto_create_subnetworks = false
}
resource "google_compute_subnetwork" "subnet" {
  name          = "documind-subnet"
  network       = google_compute_network.vpc.id
  region        = var.region
  ip_cidr_range = "10.20.0.0/20"
  private_ip_google_access = true
}
resource "google_vpc_access_connector" "conn" {
  name          = "documind-vpc"
  region        = var.region
  network       = google_compute_network.vpc.name
  ip_cidr_range = "10.8.0.0/28"
  min_instances = 2
  max_instances = 6
}
'''

REGISTRY_TF = '''
resource "google_artifact_registry_repository" "docker" {
  repository_id = "documind"
  format        = "DOCKER"
  location      = var.region
  description   = "DocuMind container images"
  cleanup_policies {
    id     = "keep-recent"
    action = "KEEP"
    most_recent_versions { keep_count = 20 }
  }
  cleanup_policies {
    id     = "delete-old"
    action = "DELETE"
    condition { older_than = "2592000s" }  # 30 days
  }
}
'''

FIRESTORE_TF = '''
resource "google_firestore_database" "main" {
  project     = var.project_id
  name        = "(default)"
  location_id = var.india_region   # DPDPA-aligned data residency
  type        = "FIRESTORE_NATIVE"
  concurrency_mode                = "OPTIMISTIC"
  app_engine_integration_mode     = "DISABLED"
  point_in_time_recovery_enablement = "POINT_IN_TIME_RECOVERY_ENABLED"
  delete_protection_state         = "DELETE_PROTECTION_ENABLED"
}
'''
for name, content in [('network.tf', NETWORK_TF), ('registry.tf', REGISTRY_TF), ('firestore.tf', FIRESTORE_TF)]:
    with open(name, 'w') as f: f.write(content)
print('network.tf + registry.tf + firestore.tf written')

## Cell 5: GCS buckets (uploads, TTS cache, audit log)

In [ ]:
STORAGE_TF = '''
resource "google_storage_bucket" "uploads" {
  name          = "${var.project_id}-uploads"
  location      = var.india_region   # in-region residency (best practice, not a DPDP mandate)
  force_destroy = false
  uniform_bucket_level_access = true
  public_access_prevention    = "enforced"
  versioning { enabled = true }
  lifecycle_rule {
    condition { age = 90 }
    action    {
      type          = "SetStorageClass"
      storage_class = "NEARLINE"
    }
  }
  lifecycle_rule {
    condition { age = 365 }
    action    { type = "Delete" }
  }
  cors {
    origin          = ["https://documind.example.com"]
    method          = ["GET", "PUT", "POST"]
    response_header = ["Content-Type"]
    max_age_seconds = 3600
  }
}

resource "google_storage_bucket" "tts_cache" {
  name          = "${var.project_id}-tts-cache"
  location      = var.india_region
  uniform_bucket_level_access = true
  lifecycle_rule {
    condition { age = 30 }
    action    { type = "Delete" }
  }
}

resource "google_storage_bucket" "audit" {
  name          = "${var.project_id}-audit"
  location      = var.india_region
  uniform_bucket_level_access = true
  retention_policy {
    retention_period = 157680000   # 5 years -- audit/RBI retention best practice (DPDP Act sets no fixed number)
    is_locked        = true
  }
}

# UI can read/write uploads + TTS cache only
resource "google_storage_bucket_iam_member" "ui_uploads" {
  bucket = google_storage_bucket.uploads.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_storage_bucket_iam_member" "ui_tts" {
  bucket = google_storage_bucket.tts_cache.name
  role   = "roles/storage.objectAdmin"
  member = "serviceAccount:${google_service_account.ui.email}"
}
'''
with open('storage.tf', 'w') as f: f.write(STORAGE_TF)
print('storage.tf written')

## Cell 6: Secret Manager (LiteLLM master key, cookie secret, OAuth)

In [ ]:
SECRETS_TF = '''
variable "secret_names" {
  type    = set(string)
  default = [
    "litellm-master-key",
    "cookie-secret",
    "oauth-client-id",
    "oauth-client-secret",
    "openai-fallback-key",
  ]
}

resource "google_secret_manager_secret" "s" {
  for_each  = var.secret_names
  secret_id = each.value
  replication { auto {} }
}

resource "google_secret_manager_secret_iam_member" "ui_access" {
  for_each  = google_secret_manager_secret.s
  secret_id = each.value.id
  role      = "roles/secretmanager.secretAccessor"
  member    = "serviceAccount:${google_service_account.ui.email}"
}
resource "google_secret_manager_secret_iam_member" "api_access" {
  for_each  = google_secret_manager_secret.s
  secret_id = each.value.id
  role      = "roles/secretmanager.secretAccessor"
  member    = "serviceAccount:${google_service_account.api.email}"
}
'''
with open('secrets.tf', 'w') as f: f.write(SECRETS_TF)
print('secrets.tf written')
print()
print('After apply: add the ACTUAL secret value with')
print('  echo -n "$(openssl rand -hex 32)" | gcloud secrets versions add cookie-secret --data-file=-')
print('  gcloud secrets versions add oauth-client-id --data-file=client-id.txt')

## Cell 7: Billing budget + alerts

In [ ]:
BUDGET_TF = '''
data "google_billing_account" "acct" {
  billing_account = var.billing_account_id
}

resource "google_billing_budget" "documind" {
  billing_account = data.google_billing_account.acct.id
  display_name    = "DocuMind monthly budget"

  budget_filter {
    projects = ["projects/${var.project_id}"]
  }

  amount {
    specified_amount {
    currency_code = "USD"
    units         = "500"
  }
  }

  threshold_rules { threshold_percent = 0.5 }
  threshold_rules { threshold_percent = 0.8 }
  threshold_rules { threshold_percent = 1.0 }
  threshold_rules {
    threshold_percent = 1.2
    spend_basis       = "FORECASTED_SPEND"
  }

  all_updates_rule {
    monitoring_notification_channels = var.alert_channels
    disable_default_iam_recipients   = false
    pubsub_topic                     = google_pubsub_topic.budget_alerts.id
  }
}

resource "google_pubsub_topic" "budget_alerts" {
  name = "documind-budget-alerts"
}

variable "billing_account_id" { type = string }
variable "alert_channels"     {
  type    = list(string)
  default = []
}
'''
with open('budget.tf', 'w') as f: f.write(BUDGET_TF)
print('budget.tf written')
print()
print('Thresholds hit at 50/80/100% actual + 120% forecasted.')
print('Pub/Sub topic lets you wire a Cloud Function that scales Cloud Run min_instances to 0 on breach.')

## Cell 8: Apply + smoke test + what the next lesson inherits

In [ ]:
APPLY = '''
# Dry-run first
terraform init -reconfigure -backend-config="bucket=documind-ai-YOUR-ID-tfstate"
terraform plan -out=tfplan \\
  -var=project_id=documind-ai-YOUR-ID \\
  -var=billing_account_id=YOUR-BILLING-ID \\
  -var='admin_emails=["alice@documind.ai"]'

# Apply when plan is green
terraform apply tfplan

# Smoke tests
gcloud iam service-accounts list --filter="email~documind-.*-sa"
gcloud artifacts repositories describe documind --location=us-central1
gcloud firestore databases list
gsutil ls -b gs://documind-ai-YOUR-ID-uploads
gcloud secrets list --filter=name~litellm
gcloud billing budgets list --billing-account=YOUR-BILLING-ID
'''
print(APPLY)
print()
INHERITS = {
    '12.2 RAG API': 'documind-api-sa + VPC connector + Firestore + Vector Search index',
    '12.3 Admin Dashboard': 'documind-admin-sa + audit bucket + budget pubsub topic',
    '12.4 Streamlit UI': 'documind-ui-sa (self-impersonation set), uploads + TTS cache buckets, all 5 secrets',
    'Operations': 'Budget alerts route to Pub/Sub; wire Cloud Function to degrade service on 100% breach',
}
print('EVERYTHING DOWNSTREAM INHERITS:')
for lesson, what in INHERITS.items():
    print(f'  {lesson:30} <- {what}')

## ✅ Lesson 12.1 Complete!

- ✅ Single `gcloud services enable` enables all 17 APIs the capstone needs
- ✅ GCS-backed Terraform state with versioning; no local state files
- ✅ Three least-privilege service accounts (ui / api / admin) with scoped roles
- ✅ Self-impersonation grant on ui-sa (required for V4 signed URLs on Cloud Run)
- ✅ VPC + VPC Access Connector for private egress to LiteLLM gateway
- ✅ Artifact Registry with cleanup policy (keep last 20, delete &gt; 30d)
- ✅ Firestore Native in `asia-south1` with PITR + delete protection (DPDPA-aligned)
- ✅ Three GCS buckets: uploads (lifecycle + CORS), TTS cache (30d), audit (5y locked retention)
- ✅ Five Secret Manager secrets with accessor bindings to ui + api
- ✅ Billing budget with 50/80/100/120% thresholds + Pub/Sub alert topic

**Next: Lesson 12.2 — RAG API Backend (FastAPI on Cloud Run)**